# 第 E01 章 MTX 格式导入练习

## 学习目标

认识 10X MTX 的三文件结构，并完成导入练习。

## 为什么做与怎样做

读取 barcodes、features 和稀疏表达矩阵，合并两个练习样本。练习到导入和对象检查为止。

前置章节：无。运行前请完成项目环境准备。


In [1]:
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
ctx = start_chapter("E01")


第 E01 章：MTX 格式导入练习
结果目录：results/E01_mtx_import/20260915T080308_5087b8


另外一种常见格式10X数据格式读取演示

In [2]:
import gzip
for filename in ["barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz"]:
    print("\n" + filename)
    with gzip.open(ROOT / "data/mtx/F30" / filename, "rt") as handle:
        for _, line in zip(range(5), handle):
            print(line.rstrip())

samples = {
    "M66": ROOT / "data/mtx/M66",  # 文件夹路径
    "F30": ROOT / "data/mtx/F30",   # 文件夹路径
}
adatas = {}

for sample_id, folder_path in samples.items():
    # 读取10X mtx格式数据
    sample_adata = sc.read_10x_mtx(
        folder_path,          # 包含三个文件的文件夹路径
        var_names='gene_symbols',  # 使用基因符号作为变量名，['gene_symbols', 'gene_ids']（默认：'gene_symbols'）
        cache=False,           # 可选：缓存以提高后续读取速度
        gex_only=True,          #仅保留‘基因表达’数据，忽略其他特征类型，例如‘抗体捕获’、‘CRISPR引导捕获’或‘自定义’数据。对于多组学数据
        prefix=None
    )
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

# 合并所有数据
adata_mtx = ad.concat(adatas, label="samples")
adata_mtx.obs_names_make_unique()
print(adata_mtx.obs["samples"].value_counts())


# 如果使用的是mtx格式，记得将下面注释的代码运行
# adata = adata_mtx.copy()



barcodes.tsv.gz
AAACCTGAGACGCTTT-1
AAACCTGAGAGACGAA-1
AAACCTGAGCTACCTA-1
AAACCTGAGGATGGAA-1
AAACCTGCAAACTGCT-1

features.tsv.gz
ENSG00000243485	MIR1302-2HG	Gene Expression
ENSG00000237613	FAM138A	Gene Expression
ENSG00000186092	OR4F5	Gene Expression
ENSG00000238009	AL627309.1	Gene Expression
ENSG00000239945	AL627309.3	Gene Expression

matrix.mtx.gz
%%MatrixMarket matrix coordinate integer general
%metadata_json: {"format_version": 2, "software_version": "3.1.0"}
33538 8392 9994017
33509 1 14
33507 1 2


samples
F30    8392
M66    7710
Name: count, dtype: int64


/data/home/heqingchuan/workdir/19_方超老师合作_单细胞_gpt6/sc_RNA/sc_RNA_basic_course/.envs/sc_rna/lib/python3.12/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [3]:
# 三个文件的详细解释
# 1. barcodes.tsv.gz - 细胞条形码文件
# 含义
# 包含每个细胞的唯一标识符（barcode）
# 每行代表一个细胞样本
# 总行数 = 细胞数量
# 2. features.tsv.gz - 基因特征文件
# 第一列（Gene ID）：
# Ensembl数据库中的唯一标识符
# 格式：ENSGXXXXXXXXXXXX
# 用于基因组浏览器、数据库查询
# 第二列（Gene Symbol）：
# 基因的常用名称/符号
# 更容易理解的名称（如TP53, ACTB）
# 第三列（Feature Type）：
# 特征类型标识：
# Gene Expression：标准mRNA表达
# 3. matrix.mtx.gz - 表达矩阵文件（核心数据）
# 三个数字（空格分隔）：
# 第1列：
# 行索引：基因的索引（从1开始）
# 第2列：
# 列索引：细胞的索引（从1开始）
# 第3列：
# 值：UMI计数（表达量）


In [4]:
print(adata_mtx.X)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 22921069 stored elements and shape (16102, 33538)>
  Coords	Values
  (0, 55)	1.0
  (0, 70)	1.0
  (0, 89)	1.0
  (0, 154)	19.0
  (0, 201)	1.0
  (0, 245)	1.0
  (0, 251)	1.0
  (0, 345)	1.0
  (0, 387)	1.0
  (0, 465)	2.0
  (0, 493)	48.0
  (0, 504)	1.0
  (0, 515)	5.0
  (0, 518)	6.0
  (0, 526)	1.0
  (0, 531)	1.0
  (0, 534)	1.0
  (0, 541)	1.0
  (0, 546)	1.0
  (0, 559)	2.0
  (0, 561)	7.0
  (0, 567)	1.0
  (0, 609)	1.0
  (0, 618)	1.0
  (0, 627)	1.0
  :	:
  (16101, 32792)	1.0
  (16101, 32809)	1.0
  (16101, 32953)	1.0
  (16101, 33131)	2.0
  (16101, 33133)	1.0
  (16101, 33207)	1.0
  (16101, 33285)	3.0
  (16101, 33297)	2.0
  (16101, 33322)	1.0
  (16101, 33359)	1.0
  (16101, 33376)	1.0
  (16101, 33394)	1.0
  (16101, 33397)	1.0
  (16101, 33406)	1.0
  (16101, 33496)	5.0
  (16101, 33497)	12.0
  (16101, 33498)	18.0
  (16101, 33499)	13.0
  (16101, 33501)	4.0
  (16101, 33502)	7.0
  (16101, 33503)	5.0
  (16101, 33504)	1.0
  (16101, 33505)	9.0
  (16

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [5]:
ctx.table("mtx_sample_counts", adata_mtx.obs["samples"].value_counts().rename("n_cells"))
ctx.finish(adata_mtx, {"purpose": "MTX 导入练习，不进入 H5 分析主线"})

本章计算完成。请阅读本次图表和表格，再更新本章解读。


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：如果 features 同时包含 Gene Expression 和其他模态，gex_only 会产生什么影响？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。